### Batch Log

In [1]:
import pandas as pd 
import numpy as np
np.random.seed(42)

In [2]:
n_batches=3000
products={"GastroEase_CR":"Coating", "CardioStat_IR":"Mixing", "NeuroCalm_IR":"Granualation"}
lines=["Line_1","Line_2"]
operators=["Operator_A","Operator_B","Operator_C","Operator_D"]

In [3]:
batch_ids=[f"BATCH_{i:04d}" for i in range(1,n_batches+1)]
base=pd.Timestamp("2023-01-01 06:00:00")

timegap=np.random.randint(low=8, high=11, size=n_batches-1)
initial_timing=[base]
for i in timegap:
    initial_timing.append(initial_timing[-1]+pd.Timedelta(hours=i))

batch_timings=pd.to_datetime(initial_timing)

In [4]:
batch_log=pd.DataFrame({"Batch_ID":batch_ids, 
                        "Product_ID":np.random.choice(list(products.keys()), n_batches),
                        "Line_ID":np.random.choice(lines, n_batches), 
                        "Operator":np.random.choice(operators, n_batches),
                        "StartTimestamp":batch_timings})

In [5]:
batch_log

,Batch_ID,Product_ID,Line_ID,Operator,StartTimestamp
0,BATCH_0001,CardioStat_IR,Line_2,Operator_A,2023-01-01 06:00:00
1,BATCH_0002,CardioStat_IR,Line_2,Operator_B,2023-01-01 16:00:00
2,BATCH_0003,CardioStat_IR,Line_2,Operator_B,2023-01-02 00:00:00
3,BATCH_0004,NeuroCalm_IR,Line_2,Operator_D,2023-01-02 10:00:00
4,BATCH_0005,NeuroCalm_IR,Line_1,Operator_A,2023-01-02 20:00:00
...,...,...,...,...,...
2995,BATCH_2996,GastroEase_CR,Line_2,Operator_D,2026-01-25 03:00:00
2996,BATCH_2997,GastroEase_CR,Line_1,Operator_A,2026-01-25 13:00:00
2997,BATCH_2998,CardioStat_IR,Line_1,Operator_C,2026-01-25 21:00:00
2998,BATCH_2999,NeuroCalm_IR,Line_2,Operator_C,2026-01-26 05:00:00


### Process Parameters

In [6]:
param_specs={"Mixing":{
                    "Mixing_Speed_RPM":(120,180),
                    "Mixing_Time_Min":(10,20),
                    "Blender_load_%":(60,75)
                    },
             "Granualation":{  
                    "Granulation_Temp_C": (55, 70),
                    "Granulation_Moisture_%": (2.0, 4.0),
                    "Granulation_Time_Min": (15, 30)
                 },
             "Coating":{
                    "Coating_Spray_Rate": (5, 10),
                    "Inlet_Temp_C": (45, 60),
                    "Coating_Time_Min": (20, 40)
             }}

In [7]:
def generate_param_value(low, high, oos_prob=0.15):
    if np.random.rand()<=oos_prob:
        return np.random.uniform(low*0.80, high*1.20)
    else:
        return np.random.uniform(low, high)
    
rows=[]
    
for _,row in batch_log.iterrows():
    for step, params in param_specs.items():
        for param, (low, high) in params.items():
            rows.append({
                "Batch_ID": row["Batch_ID"],
                "Step":step,
                "Param_Name": param,
                "Param_Value": round(generate_param_value(low,high),4)
            })

process_parameters=pd.DataFrame(rows)

In [8]:
process_parameters

,Batch_ID,Step,Param_Name,Param_Value
0,BATCH_0001,Mixing,Mixing_Speed_RPM,124.9493
1,BATCH_0001,Mixing,Mixing_Time_Min,12.5048
2,BATCH_0001,Mixing,Blender_load_%,64.1951
3,BATCH_0001,Granualation,Granulation_Temp_C,62.5640
4,BATCH_0001,Granualation,Granulation_Moisture_%,2.4827
...,...,...,...,...
26995,BATCH_3000,Granualation,Granulation_Moisture_%,3.9623
26996,BATCH_3000,Granualation,Granulation_Time_Min,25.4292
26997,BATCH_3000,Coating,Coating_Spray_Rate,6.1295
26998,BATCH_3000,Coating,Inlet_Temp_C,47.9517


### Quality Table with OOS Flags

In [9]:
# This table has been created for reference and OOS marking
pivoted_params= process_parameters.pivot_table(
    index="Batch_ID",
    columns="Param_Name",
    values="Param_Value"
).reset_index()

merged_log_param=batch_log.merge(pivoted_params,on="Batch_ID")

In [10]:
merged_log_param

,Batch_ID,Product_ID,Line_ID,Operator,StartTimestamp,Blender_load_%,Coating_Spray_Rate,Coating_Time_Min,Granulation_Moisture_%,Granulation_Temp_C,Granulation_Time_Min,Inlet_Temp_C,Mixing_Speed_RPM,Mixing_Time_Min
0,BATCH_0001,CardioStat_IR,Line_2,Operator_A,2023-01-01 06:00:00,64.1951,7.7535,32.0761,2.4827,62.5640,20.5537,59.3939,124.9493,12.5048
1,BATCH_0002,CardioStat_IR,Line_2,Operator_B,2023-01-01 16:00:00,70.0006,7.6885,32.9209,2.8915,63.7867,18.9745,56.7596,122.4041,13.8348
2,BATCH_0003,CardioStat_IR,Line_2,Operator_B,2023-01-02 00:00:00,56.6179,10.4212,38.8662,3.8597,68.1738,22.3567,58.4942,175.9589,17.2952
3,BATCH_0004,NeuroCalm_IR,Line_2,Operator_D,2023-01-02 10:00:00,60.5145,8.0295,31.1342,2.7301,57.3226,24.3476,53.6096,160.6256,11.3101
4,BATCH_0005,NeuroCalm_IR,Line_1,Operator_A,2023-01-02 20:00:00,69.7215,8.4464,36.7114,2.6197,70.1598,14.1825,48.8420,140.1366,14.3277
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2995,BATCH_2996,GastroEase_CR,Line_2,Operator_D,2026-01-25 03:00:00,64.2908,6.9663,29.3519,2.1360,61.2951,29.0460,46.3982,144.6094,16.0971
2996,BATCH_2997,GastroEase_CR,Line_1,Operator_A,2026-01-25 13:00:00,65.1828,9.5451,24.9349,2.6168,59.8313,20.7728,48.5186,136.2273,13.4538
2997,BATCH_2998,CardioStat_IR,Line_1,Operator_C,2026-01-25 21:00:00,67.1262,7.2713,26.0878,2.8179,60.3138,13.5630,46.6020,158.9730,14.0584
2998,BATCH_2999,NeuroCalm_IR,Line_2,Operator_C,2026-01-26 05:00:00,89.7713,5.9213,36.8964,2.4622,57.5501,16.7844,53.6137,131.6010,11.9332


In [11]:
def quality_marker(row):
    assay=100
    impurity=0.2
    dissolution=90
    hardness=7

    if row["Product_ID"]=="GastroEase_CR":
        if row["Coating_Spray_Rate"] > 9 and row["Inlet_Temp_C"] < 48:
            dissolution -= np.random.uniform(5, 10)
    
    if row["Product_ID"]=="CardioStat_IR":
        if row["Mixing_Speed_RPM"] < 130:
            assay -= np.random.uniform(3, 6)

    if row["Product_ID"]=="NeuroCalm_IR":
        if row["Granulation_Temp_C"] > 68 or row["Granulation_Moisture_%"] > 3.5:
            impurity += np.random.uniform(0.2, 0.5)

    return assay, impurity, dissolution, hardness

quality_list=[]

for _, row in merged_log_param.iterrows():
    assay, impurity, dissolution, hardness = quality_marker(row)

    oos= int((assay <= 98) or (impurity > 0.4) or (dissolution < 85) or (hardness < 5 or hardness > 10)) # OOS Flagging 1 or 0

    quality_list.append({
        "Batch_ID": row["Batch_ID"],
        "Assay":assay,
        "Dissolution":dissolution,
        "Impurity":impurity,
        "Hardness": hardness,
        "OOS_Flag":oos
    })

Quality_table=pd.DataFrame(quality_list)

In [12]:
Quality_table

,Batch_ID,Assay,Dissolution,Impurity,Hardness,OOS_Flag
0,BATCH_0001,96.145266,90.0,0.200000,7,1
1,BATCH_0002,95.873683,90.0,0.200000,7,1
2,BATCH_0003,100.000000,90.0,0.200000,7,0
3,BATCH_0004,100.000000,90.0,0.200000,7,0
4,BATCH_0005,100.000000,90.0,0.508155,7,1
...,...,...,...,...,...,...
2995,BATCH_2996,100.000000,90.0,0.200000,7,0
2996,BATCH_2997,100.000000,90.0,0.200000,7,0
2997,BATCH_2998,100.000000,90.0,0.200000,7,0
2998,BATCH_2999,100.000000,90.0,0.200000,7,0


### DF to CSV save

In [13]:
batch_log.to_csv("../data/raw/batch_log.csv", index=False)
Quality_table.to_csv("../data/raw/quality_table.csv", index=False)
process_parameters.to_csv("../data/raw/process_parameters.csv", index=False)